# 01 — EDA i kontrakt danych (Sprint 1)

Cel notebooka: poznać dane wybranego datasetu i podjąć decyzje, które trafią do kodu:

- **input normalization** — reguły w `normalize_input()` (`src/asi_project/data.py`),
- **training-only filtering** — reguły w `filter_training_rows()` (jeśli są potrzebne),
- **typy cech** — listy `features.numeric` i `features.categorical` w `project.yaml`,
- **metryka główna** — uzasadnienie w karcie projektu w README.

Jak pracować:

- uruchamiajcie notebook od góry (Restart & Run All); po zmianie `project.yaml` uruchomcie
  ponownie komórkę ustawień,
- komórki oznaczone **TODO** uzupełnia zespół; odpowiedzi tekstowe wpisujcie w miejsce
  `*(odpowiedź)*`,
- nie zmieniajcie kontraktu prowadzącej (`src/asi_project/datasets.yaml`).

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
# DONE
# Obserwacje: które kolumny mają nieoczekiwany typ? Gdzie są braki?
# Które kolumny liczbowe są w rzeczywistości kategoriami?

df_nonempty = df_raw.dropna(how="all")
print(f"Wiersze w całości NaN: {len(df_raw) - len(df_nonempty)} ==> nowe n = {len(df_nonempty)}")
same_mask = df_nonempty["Credit Score"].isna().equals(df_nonempty["Annual Income"].isna())
print("maska NaN: Credit Score == Annual Income:", same_mask)

print("\nObserwacje po usunięciu wierszy w całości NaN:")
numeric = df_nonempty.select_dtypes("number")
text = df_nonempty.select_dtypes("object")


def levels(col):
    n = col.nunique()
    if n <= 4:
        return col.value_counts().to_dict()
    elif n <= 20:
        head = col.value_counts().head(2)
        tail = col.value_counts().tail(-2)
        return f"{head.to_dict()}, ... ({len(tail)} others)"
    else:
        return "..."


observations = pd.DataFrame(
    {
        "dtype": df_nonempty.dtypes.astype(str),
        "NaN": df_nonempty.isna().sum(),
        "NaN %": (df_nonempty.isna().mean() * 100).round(1),
        "unikalne": df_nonempty.nunique(),
        "całkowite": numeric.apply(lambda col: (col.dropna() % 1 == 0).all()),
        "poziomy": text.apply(levels),
    },
).reindex(df_nonempty.columns)
observations

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
print("Klasa pozytywna:", repr(contract.positive_class))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()

**DONE — metryka główna (2–3 zdania):** którą metrykę (accuracy, precision, recall, F1)
wybieracie jako główną i dlaczego, biorąc pod uwagę udział klasy pozytywnej i koszt błędów
w waszym problemie? To samo uzasadnienie wpiszcie do karty projektu w README.

**Metryka główna: F1 dla klasy `Charged Off`** (udział klasy pozytywnej: 0,226)

| metryka | ocena |
|---|---|
| accuracy | odrzucona: model trywialny przewidujący "zawsze `Fully Paid`" daje 0,774 bez żadnej wiedzy |
| recall | ważna (FN = niewykryty kredyt niespłacony, koszt ~ zainwestowany kapitał), ale sama metryka jest maksymalizowana przez "zawsze `Charged Off`" |
| precision | ważna (FP = odrzucony dobry klient, koszt ~ utracona marża), ale sama nagradza bardzo ostrożny model |
| **F1** | **wybrana**: łączy recall i precision dla klasy pozytywnej, odporna na niezbalansowanie 77/23 |

## 5. EDA

**DONE:** co najmniej dwie analizy — jedna dla cechy numerycznej, jedna dla kategorycznej.
Wybierajcie wyłącznie spośród ALLOWED FEATURES.

In [ ]:
# Cecha numeryczna: Credit Score w podziale na klasy targetu
df_nonempty.groupby(contract.target)["Credit Score"].describe()

In [ ]:
# Cecha kategoryczna: udział klasy pozytywnej w kategoriach Term
pd.crosstab(df_nonempty["Term"], df_nonempty[contract.target], normalize="index").round(3)

In [ ]:
pd.crosstab(df_nonempty["Term"], df_nonempty[contract.target], normalize="columns").round(3)

**DONE — wnioski z EDA (3–5 zdań):**

- `Credit Score`: `Fully Paid` ma max 751, a `Charged Off` max 7 510 -> część wyników w `Charged Off` jest zapisana w innej skali.
- `Long Term` ma więcej `Charged Off` niż `Short Term` (0,301 vs 0,198).
- Klasa pozytywna `Charged Off` to (niewielka część) 0,226 danych.
- 514 wierszy w całości NaN; `Credit Score` i `Annual Income` mają braki w tych samych wierszach. Prawdopodobnie bez przychodu nie można było obliczyć punktów.
- `Home Ownership` ma dwa zapisy jednej kategorii: `HaveMortgage` i `Home Mortgage`.
- `Current Loan Amount` występuje sztuczna wartość 99999999 wskazująca na braki danych

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")

**DONE — reguły input normalization:** jakie reguły dopisujecie do `normalize_input()` i dlaczego
są bezpieczne przy predykcji pojedynczego rekordu? Po zmianie funkcji uruchomcie `pytest`.

- W `Purpose` zamieniamy `other` na `Other`, bo to ta sama kategoria zapisana na dwa sposoby.
- W `Credit Score` dzielimy przez 10 wartości czterocyfrowe, bo to zwykłe wyniki z dopisanym zerem: po podzieleniu mają ten sam zakres co pozostałe.
- W `Home Ownership`  Ujednolicono kategorię HaveMortgage do Home Mortgage
- W `Current Loan Amount` zmieniono 99999999 na braki danych (NaN), aby zapobiec wyciekowi informacji
- Reguły są bezpieczne: dziłają na tylko jednym polu, nie używają targetu ani statystyk, są idempotentne.

In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")

**DONE — decyzja o training-only filtering:** czy usuwacie dodatkowo jakieś rekordy
(np. dokładne duplikaty)? Dlaczego tak albo dlaczego nie?

- Tak, usuwamy dokładne duplikaty wierszy, bo ten sam kredyt zapisany kilka razy ma w treningu większą wagę.
  Na dodatek, może trafić jednocześnie do zbioru treningowego i testowego.

## 8. Typy cech → `project.yaml`

**DONE:** wpiszcie do `project.yaml` listy `features.numeric` i `features.categorical`
(wyłącznie ALLOWED FEATURES), zapiszcie plik i uruchomcie ponownie komórkę ustawień oraz
komórkę poniżej.

In [ ]:
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

## 9. Podsumowanie decyzji

**DONE — w 3–5 punktach:** decyzje o input normalization, training-only filtering, typach cech
i metryce głównej. Te decyzje będą punktem wyjścia do ADR-1 w Sprincie 2.

- Input normalization: w `Purpose` zamieniamy `other` na `Other`, a w `Credit Score` dzielimy przez 10 wartości czterocyfrowe, w Home Ownership ujednolicamy HaveMortgage do Home Mortgage, a w Current Loan Amount zastępujemy sztuczną wartość 99999999 brakami danych
- Training-only filtering: oprócz rekordów bez targetu usuwamy dokładne duplikaty wierszy.
- Typy cech: 4 kolumny tekstowe są kategoryczne, pozostałe 12 numeryczne.
- Metryka główna: F1 dla klasy `Charged Off`, bo accuracy przy niezbalansowanych klasach nagradza model, który zawsze przewiduje `Fully Paid`.